In [0]:
customers_df =spark.table("bronze.customers") 
customers_df.show(5)
customers_df.printSchema()

In [0]:
# Basic Data Quality checks
from pyspark.sql import functions as F
customers_df.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("customer_id").alias("distinct_customers")
).show()

In [0]:
# Check NULLs
customers_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in customers_df.columns
]).show()

In [0]:
# Create Clean Customer Table
from pyspark.sql import functions as F
silver_customers = (
    customers_df
    .dropDuplicates(["customer_id"])
    .withColumn(
        "customer_city",
        F.initcap(F.trim(F.col("customer_city")))
    )
    .withColumn(
        "customer_state",
        F.upper(F.col("customer_state"))
    )
)

In [0]:
# Save Silver table
(
    silver_customers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.customers")
)

In [0]:
# VALIDATE 
spark.sql("""
SELECT COUNT(*)
FROM silver.customers
""").show()

In [0]:
# Read Bronze Orders
orders_df = spark.table("bronze.orders")
orders_df.show(5)
orders_df.printSchema()

In [0]:
# Creating useful Date columns
from pyspark.sql import functions as F
silver_orders = (
    orders_df
    .withColumn(
        "order_date",
        F.to_date("order_purchase_timestamp")
    )
    .withColumn(
        "order_year",
        F.year("order_purchase_timestamp")
    )
    .withColumn(
        "order_month",
        F.month("order_purchase_timestamp")
    )
    .withColumn(
        "order_month_name",
        F.date_format("order_purchase_timestamp", "MMM")
    )
)
silver_orders.printSchema()

In [0]:
# Standardize Order status
silver_orders = silver_orders.withColumn(
    "order_status",
    F.upper(F.trim(F.col("order_status")))
)
silver_orders.printSchema()

# Check Order status distribution -- for business insights.
silver_orders.groupBy("order_status") \
             .count() \
             .orderBy(F.desc("count")) \
             .show()
silver_orders.printSchema()
# save silver.orders table
(
    silver_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.orders")
)

In [0]:
# VALIDATE
spark.sql("""
SELECT
    COUNT(*) AS total_orders
FROM silver.orders
""").show()

In [0]:
order_items_df = spark.table("bronze.order_items")
order_items_df.show(5)
order_items_df.printSchema()

In [0]:
# Create Silver Transformations
# Remove duplicates , Convert shipping date to timestamp if needed , Add total item value 
from pyspark.sql import functions as F
silver_order_items = (
    order_items_df
        .dropDuplicates()
        .withColumn(
            "total_item_value",
            F.col("price") + F.col("freight_value")
        )
)
# Basic DQ Quality Check
silver_order_items.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_order_items.columns
]).show()
# Revenue Sanity check
silver_order_items.select(
    F.sum("price").alias("product_revenue"),
    F.sum("freight_value").alias("freight_revenue"),
    F.sum("total_item_value").alias("total_revenue")
).show()

# Save Table
(
    silver_order_items.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("silver.order_items")
)
spark.sql("""
SELECT COUNT(*)
FROM silver.order_items
""").show()

In [0]:
products_df = spark.table("bronze.products")
products_df.show(5)
products_df.printSchema()

# Basic Data Quality checks
from pyspark.sql import functions as F
products_df.select(F.count("*").alias("total_rows"),
                   F.countDistinct("product_id").alias("distinct products")
).show()

# NULL CHECK
products_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in products_df.columns
]).show()

In [0]:
# Create clean product Table 
#we will Remove duplicates, Replace missing categories, Standarize category names.
silver_products = products_df.drop_duplicates(["product_id"]).fillna({"product_category_name" : "UNKNOWN"}).withColumn("product_category_name",F.upper(F.trim(F.col("product_category_name")))                                                                                                        )
# Add product size metrics
silver_products = silver_products.withColumn(
    "product_volume_cm3",
    F.col("product_length_cm")
    * F.col("product_width_cm")
    * F.col("product_height_cm")
)
# Validate Categories
silver_products.groupBy(
    "product_category_name"
).count().orderBy(
    F.desc("count")
).show(20, False)

# Save Silver table
(
    silver_products.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("silver.products")
)
# Validate
spark.sql("""
SELECT COUNT(*)
FROM silver.products
""").show()


In [0]:
from pyspark.sql import functions as F
payments_df = spark.table("bronze.payments")
payments_df.show(5)
payments_df.printSchema()

In [0]:
# Silver payment transformations
# Remove Duplicates , Standarize Payment_type ,Add installment category
silver_payments = (
    payments_df
    .dropDuplicates()
    .withColumn(
        "payment_type",
        F.upper(F.trim(F.col("payment_type")))
    )
    .withColumn(
        "installment_bucket",
        F.when(F.col("payment_installments") == 1, "ONE_TIME")
         .when(F.col("payment_installments") <= 3, "SHORT_TERM")
         .when(F.col("payment_installments") <= 6, "MEDIUM_TERM")
         .otherwise("LONG_TERM")
    )
)
# validate payment types 
silver_payments.groupBy("payment_type") \
               .count() \
               .orderBy(F.desc("count")) \
               .show()
# Revenue Validation
silver_payments.select(
    F.sum("payment_value").alias("total_payment_value")
).show()
print(silver_payments)

# Save table 
(
    silver_payments.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("silver.payments")
)
# Validate
spark.sql("""
SELECT COUNT(*)
FROM silver.payments
""").show()

In [0]:
from pyspark.sql import functions as F 
reviews_df = spark.table("bronze.reviews")
# reviews_df.show(5)
# reviews_df.printSchema()
# reviews_df.select("review_score").distinct().show()

# /Volumes/workspace/bronze/raw_data_volume/ecommerce_dataset/olist_order_reviews_dataset.csv
#  DQ Check 
reviews_df.select(F.count("*").alias("Total_rows"),F.countDistinct("review_id").alias("distinct_reviews")).show()
# Inspecting the error on review_score -- it should be int 
# reviews_df.select("review_score").distinct().show(20, truncate=False)
# reviews_df = reviews_df.withColumn(
#     "review_score",
#     F.col("review_score").cast("int")
# )
# reviews_df.show(10)

bad_reviews = reviews_df.filter(
    ~F.col("review_score").rlike("^[1-5]$")
)
display(bad_reviews)
# spark.sql("""
# SELECT *
# FROM bronze.reviews
# LIMIT 5
# """).show(truncate=False)

We are not working with reviews table , there is lot of discrepency in data.

In [0]:
from pyspark.sql import functions as F
# Create a clean silver reviews table 
silver_reviews = (
    reviews_df
    .filter(F.col("review_score").rlike("^[1-5]$"))
    .withColumn("review_score", F.col("review_score").cast("int"))
    .dropDuplicates(["review_id"])
    .withColumn(
        "review_sentiment",
        F.when(F.col("review_score") >= 4, "POSITIVE")
         .when(F.col("review_score") == 3, "NEUTRAL")
         .otherwise("NEGATIVE")
    )
    .withColumn(
        "review_date",
        F.to_date("review_creation_date")
    )
    .withColumn(
        "review_year",
        F.year(F.to_date("review_creation_date"))
    )
    .withColumn(
        "review_month",
        F.month(F.to_date("review_creation_date"))
    )
)

# # Review Sentiment distribution
silver_reviews.groupBy(
    "review_sentiment"
).count().show()

In [0]:
# # Save it 
# (
#     silver_reviews.write
#     .format("delta")
#     .mode("overwrite")
#     .saveAsTable("silver.reviews")
# )


In [0]:
sellers_df = spark.table("bronze.sellers")

silver_sellers = (
    sellers_df
    .dropDuplicates(["seller_id"])
)

silver_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.sellers")

    # Validate the data
spark.sql("""
SELECT COUNT(*)
FROM silver.sellers
""").show()

In [0]:
silver_translation = (
    spark.table("bronze.category_name")
         .dropDuplicates()
)

silver_translation.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.category_translation")

In [0]:
spark.table("silver.customers").printSchema()
spark.table("silver.orders").printSchema()
spark.table("silver.order_items").printSchema()
spark.table("silver.products").printSchema()
spark.table("silver.sellers").printSchema()
spark.table("silver.category_translation").printSchema()
spark.table("silver.payments").printSchema()